In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("source_file_format", "")

In [0]:
source_system=dbutils.widgets.get("source_system")
table_name=dbutils.widgets.get("table_name")
landing_timestamp=dbutils.widgets.get("landing_timestamp")
source_file_format=dbutils.widgets.get("source_file_format")

In [0]:
base_path = f"/Volumes/databricks_rg/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

try:
    entries = dbutils.fs.ls(base_path)
except Exception:
    parent_path = f"/Volumes/databricks_rg/landing/landing_volume/{source_system}/{table_name}/"
    try:
        available = [f.name.rstrip('/') for f in dbutils.fs.ls(parent_path)]
    except Exception:
        available = []
    msg = (
        f"SKIPPED: Landing path not found: {base_path} | "
        f"Available timestamps: {available} | "
        f"Check that the ADF copy activity for '{table_name}' succeeded."
    )
    print(msg)
    dbutils.notebook.exit(msg)

file_ext = ".csv" if source_file_format == "csv" else ".parquet"
file_paths = [f.path for f in entries if f.name.endswith(file_ext)]

if not file_paths:
    raise FileNotFoundError(f"No {file_ext} files found in {base_path}")

if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_paths)
else:
    landing_df = spark.read.format("parquet").load(file_paths)



In [0]:
from pyspark.sql import functions as F

In [0]:
updated_df=landing_df.withColumn("landing_timestamp",F.lit(landing_timestamp)).withColumn("insert_timestamp",F.current_timestamp())



In [0]:
spark.conf.set(
    "spark.databricks.delta.schema.autoMerge.enabled",
    "true"
)

In [0]:
updated_df.write.mode("append").saveAsTable(f"databricks_rg.bronze.{table_name}")